# 02 - genscai

The same agentic literature-research use case as the other notebooks, but built with the **genscai `Corpus` helper** instead of wiring tools by hand. `genscai.corpus.Corpus` packages the whole pattern — search → relevance-gate → persist → read — as three ready-made agent tools over a vector-backed store, on top of AIMU.

Under the hood it leans on three AIMU capabilities: `ToolContext` (so the tools reach the store and DOI cache without module globals), `EvaluatorOptimizer(verdict_schema=...)` (a typed critic verdict instead of matching a `PASS` string), and `aimu.pretty_print` (renders the streamed run). Compare with **01 - AIMU**, which wires the same three tools by hand.

Requires a local Ollama server with a tool-capable chat model and `nomic-embed-text` for embeddings (see the folder README).

In [ ]:
import os

import aimu
from aimu.agents import Agent, EvaluatorOptimizer
from pydantic import BaseModel

from genscai import research, paths
from genscai.corpus import Corpus

MODEL = f"ollama:{os.environ.get('GENSCAI_AGENT_MODEL', 'qwen3.6:27b')}"

RESEARCH_QUESTION = (
    "What interventions and control strategies have recent preprints proposed or evaluated for "
    "dengue outbreaks, and what evidence do they report?"
)

## The corpus

One constructor wires `search_preprints`, `save_relevant_paper`, and `read_saved_papers` over a persistent semantic store. The DOI cache that lets `save_relevant_paper` persist a paper the agent only named by DOI is held inside `corpus`, not in a module global.

In [ ]:
corpus = Corpus(
    search_fn=lambda q: research.search_medrxiv(q, max_results=5) + research.search_biorxiv(q, max_results=3),
    persist_path=str(paths.output / "literature_research" / "genscai_corpus_store"),
    embedding_model="ollama:nomic-embed-text",
)
print("tools:", [t.__name__ for t in corpus.tools])

## Researcher

The researcher agent gets the corpus's three tools. `aimu.pretty_print` renders the streamed run — each tool call is flagged and the generated text streams inline — and returns the final text.

In [ ]:
RESEARCHER_SYSTEM = (
    "You are a research assistant building a cited literature review grounded in a curated corpus. "
    "Workflow: (1) call read_saved_papers to see what is curated; (2) use search_preprints, always "
    "including the key topic term (e.g. 'dengue') in queries; (3) for each candidate, if its abstract "
    "is plausibly relevant call save_relevant_paper(doi), erring toward saving; (4) before synthesizing "
    "you MUST call read_saved_papers and base the synthesis only on those papers, citing each by title "
    "and DOI. Never claim no papers were found if the store is non-empty. Be concise."
)

researcher = Agent(
    aimu.client(MODEL),
    system_message=RESEARCHER_SYSTEM,
    tools=corpus.tools,
    max_iterations=12,
    reset_messages_on_run=True,
    final_answer_prompt=(
        "Call read_saved_papers, then write the final cited synthesis based only on those saved papers. "
        "Do not claim there are no papers if the store is non-empty."
    ),
    name="researcher",
)

aimu.pretty_print(researcher.run(RESEARCH_QUESTION, stream=True))

## Critic and feedback loop

The critic returns a typed `Verdict(passed, feedback)`. `EvaluatorOptimizer(verdict_schema=Verdict)` reads `passed` to accept and `feedback` to drive a revision — no `PASS` substring matching, so the critic prompt is written for humans, not for a magic keyword.

In [ ]:
class Verdict(BaseModel):
    passed: bool
    feedback: str = ""


critic = Agent(
    aimu.client(MODEL),
    system_message=(
        "You review a literature synthesis built from a curated corpus of saved preprints. Set "
        "passed=true when it answers the question, cites specific papers (title + DOI), and reports "
        "evidence from them; otherwise set passed=false and put the single most important missing piece "
        "in feedback. Do not demand topics outside the saved corpus, and do not penalize a focused answer."
    ),
    max_iterations=1,
    reset_messages_on_run=True,
    name="critic",
)

review = EvaluatorOptimizer(generator=researcher, evaluator=critic, max_rounds=2, verdict_schema=Verdict)
synthesis = review.run(RESEARCH_QUESTION)
print(synthesis)

## Saved corpus

`corpus.saved_titles` lists what the relevance gate kept. The store persists at `persist_path`, so a later run reuses it.

In [ ]:
print(f"{len(corpus.saved_titles)} papers saved:")
for title in corpus.saved_titles:
    print(" -", title[:80])